# Phase 2 - fused attention in Triton (the kernel win)

Replaces the **memory-bound** naive attention (materializes the full `[seq, seq]` score matrix in HBM) with a **FlashAttention-style Triton kernel** that never writes it. Autotuned for block size / warps / pipeline stages. Runs on **Colab's free T4**.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**. Then Runtime -> Run all.

The notebook **gates on correctness first** and refuses to report a speedup unless the kernel matches PyTorch. If a cell errors, paste the output back.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, triton
print('torch', torch.__version__, '| triton', triton.__version__, '| cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Set Runtime -> T4 GPU'

## 1. The kernel (autotuned)

In [ ]:
# --- FlashAttention forward in Triton (same as kernels/triton/fused_attention.py) ---
import torch, triton
import triton.language as tl

def _configs():
    return [
        triton.Config({"BLOCK_M": 64,  "BLOCK_N": 64},  num_warps=4, num_stages=2),
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 64},  num_warps=4, num_stages=3),
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 64},  num_warps=8, num_stages=3),
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 128}, num_warps=8, num_stages=2),
        triton.Config({"BLOCK_M": 64,  "BLOCK_N": 64},  num_warps=4, num_stages=4),
        triton.Config({"BLOCK_M": 64,  "BLOCK_N": 32},  num_warps=4, num_stages=4),
    ]

@triton.autotune(configs=_configs(), key=["N"])
@triton.jit
def _attention_kernel(
    Q, K, V, Out, scale,
    stride_qb, stride_qh, stride_qm, stride_qd,
    stride_kb, stride_kh, stride_kn, stride_kd,
    stride_vb, stride_vh, stride_vn, stride_vd,
    stride_ob, stride_oh, stride_om, stride_od,
    H, N, D: tl.constexpr, CAUSAL: tl.constexpr,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_bh = tl.program_id(1)
    b = pid_bh // H
    h = pid_bh % H
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_d = tl.arange(0, D)
    q_base = Q + b * stride_qb + h * stride_qh
    q = tl.load(q_base + offs_m[:, None] * stride_qm + offs_d[None, :] * stride_qd,
                mask=offs_m[:, None] < N, other=0.0)
    m_i = tl.full([BLOCK_M], -float("inf"), tl.float32)
    l_i = tl.zeros([BLOCK_M], tl.float32)
    acc = tl.zeros([BLOCK_M, D], tl.float32)
    hi = (pid_m + 1) * BLOCK_M if CAUSAL else N
    k_base = K + b * stride_kb + h * stride_kh
    v_base = V + b * stride_vb + h * stride_vh
    for start_n in range(0, hi, BLOCK_N):
        offs_n = start_n + tl.arange(0, BLOCK_N)
        kt = tl.load(k_base + offs_d[:, None] * stride_kd + offs_n[None, :] * stride_kn,
                     mask=offs_n[None, :] < N, other=0.0)   # [D, BLOCK_N], pre-transposed
        s = tl.dot(q, kt) * scale
        s = tl.where(offs_n[None, :] < N, s, -float("inf"))
        if CAUSAL:
            s = tl.where(offs_m[:, None] >= offs_n[None, :], s, -float("inf"))
        m_new = tl.maximum(m_i, tl.max(s, 1))
        p = tl.exp(s - m_new[:, None])
        corr = tl.exp(m_i - m_new)
        l_i = l_i * corr + tl.sum(p, 1)
        v = tl.load(v_base + offs_n[:, None] * stride_vn + offs_d[None, :] * stride_vd,
                    mask=offs_n[:, None] < N, other=0.0)
        acc = acc * corr[:, None] + tl.dot(p.to(v.dtype), v)
        m_i = m_new
    acc = acc / l_i[:, None]
    o_base = Out + b * stride_ob + h * stride_oh
    tl.store(o_base + offs_m[:, None] * stride_om + offs_d[None, :] * stride_od,
             acc.to(Out.dtype.element_ty), mask=offs_m[:, None] < N)

def fused_attention_bhsd(q, k, v, causal=True):
    q, k, v = q.contiguous(), k.contiguous(), v.contiguous()
    B, H, N, D = q.shape
    out = torch.empty_like(q)
    grid = lambda META: (triton.cdiv(N, META["BLOCK_M"]), B * H)
    _attention_kernel[grid](
        q, k, v, out, D ** -0.5,
        q.stride(0), q.stride(1), q.stride(2), q.stride(3),
        k.stride(0), k.stride(1), k.stride(2), k.stride(3),
        v.stride(0), v.stride(1), v.stride(2), v.stride(3),
        out.stride(0), out.stride(1), out.stride(2), out.stride(3),
        H, N, D=D, CAUSAL=causal,
    )
    return out

print("kernel defined (autotuned). triton", triton.__version__)

## 2. Correctness gate (runs before any benchmark)

Fused fp16 output must match a fp32 reference **and** PyTorch SDPA, including at a non-block-multiple length (N=1000).

In [ ]:
import torch, torch.nn.functional as F

def naive_ref(q, k, v, causal=True):
    # fp32 reference for CORRECTNESS (materializes the full score matrix).
    scale = q.shape[-1] ** -0.5
    s = (q.float() @ k.float().transpose(-1, -2)) * scale
    if causal:
        N = q.shape[-2]
        m = torch.tril(torch.ones(N, N, device=q.device, dtype=torch.bool))
        s = s.masked_fill(~m, float("-inf"))
    return (torch.softmax(s, -1) @ v.float()).to(v.dtype)

torch.manual_seed(0)
ok = True
for (B, H, N, D) in [(1, 4, 128, 64), (2, 8, 512, 64), (1, 8, 1000, 64)]:  # incl. non-block-multiple N
    q = torch.randn(B, H, N, D, device="cuda", dtype=torch.float16)
    k = torch.randn(B, H, N, D, device="cuda", dtype=torch.float16)
    v = torch.randn(B, H, N, D, device="cuda", dtype=torch.float16)
    tri = fused_attention_bhsd(q, k, v, causal=True)
    ref = naive_ref(q, k, v, causal=True)
    sdpa = F.scaled_dot_product_attention(q, k, v, is_causal=True)
    d_ref = (tri.float() - ref.float()).abs().max().item()
    d_sdpa = (tri.float() - sdpa.float()).abs().max().item()
    passed = d_ref < 2e-2 and d_sdpa < 2e-2
    ok = ok and passed
    print(f"[{'OK ' if passed else 'BAD'}] B{B} H{H} N{N}: max|triton-ref|={d_ref:.4f}  max|triton-sdpa|={d_sdpa:.4f}")
assert ok, "CORRECTNESS FAILED -- do not trust the benchmark; paste this output back."
print("\nCORRECTNESS GATE: PASS")

## 3. Benchmark vs the memory-bound naive path (fp16, honest)

The naive baseline is now **fp16** (fp16 matmuls, fp32 softmax) - a fair fight, not a strawman. `sdpa` is PyTorch's own fused kernel, the ceiling. The fused kernel's advantage grows with `seq` because that's when materializing `[N,N]` scores dominates.

In [ ]:
import triton, torch, torch.nn.functional as F

def naive_fp16(q, k, v, causal=True):
    # HONEST memory-bound baseline: fp16 matmuls, fp32 softmax, full [N,N] scores.
    scale = q.shape[-1] ** -0.5
    s = (q @ k.transpose(-1, -2)) * scale
    if causal:
        N = q.shape[-2]
        m = torch.tril(torch.ones(N, N, device=q.device, dtype=torch.bool))
        s = s.masked_fill(~m, float("-inf"))
    p = torch.softmax(s.float(), -1).to(q.dtype)
    return p @ v

def bench(fn):
    return triton.testing.do_bench(fn, warmup=25, rep=100)

B, H, D = 1, 8, 64
print(f"causal attention, B={B} H={H} D={D}, fp16, T4\n")
print(f"{'seq':>6} {'naive fp16 ms':>14} {'triton ms':>10} {'sdpa ms':>9}  {'triton vs naive':>16}")
print("-" * 64)
rows = []
for N in [512, 1024, 2048, 4096]:
    q = torch.randn(B, H, N, D, device="cuda", dtype=torch.float16)
    k = torch.randn(B, H, N, D, device="cuda", dtype=torch.float16)
    v = torch.randn(B, H, N, D, device="cuda", dtype=torch.float16)
    t_naive = bench(lambda: naive_fp16(q, k, v, True))
    t_tri = bench(lambda: fused_attention_bhsd(q, k, v, True))
    t_sdpa = bench(lambda: F.scaled_dot_product_attention(q, k, v, is_causal=True))
    sp = t_naive / t_tri
    rows.append((N, sp, t_tri, t_sdpa))
    print(f"{N:>6} {t_naive:>14.3f} {t_tri:>10.3f} {t_sdpa:>9.3f}  {sp:>14.2f}x")
print()
best = max(rows, key=lambda r: r[1])
print(f"Best vs naive fp16: {best[1]:.2f}x at seq={best[0]}   (triton {best[2]:.3f} ms, sdpa {best[3]:.3f} ms)")
print("Report the number YOU measured. If <1.0x anywhere, that regime is a loss -- say so.")

## 4. Profile: where the time goes

CUDA-op breakdown. Naive shows separate matmul + softmax + elementwise ops (the materialized-scores traffic); fused collapses them into one `_attention_kernel`.

In [ ]:
from torch.profiler import profile, ProfilerActivity

B, H, N, D = 1, 8, 2048, 64
q = torch.randn(B, H, N, D, device="cuda", dtype=torch.float16)
k = torch.randn(B, H, N, D, device="cuda", dtype=torch.float16)
v = torch.randn(B, H, N, D, device="cuda", dtype=torch.float16)

for name, fn in [("naive_fp16", lambda: naive_fp16(q, k, v, True)),
                 ("triton_fused", lambda: fused_attention_bhsd(q, k, v, True))]:
    for _ in range(10):
        fn()
    torch.cuda.synchronize()
    with profile(activities=[ProfilerActivity.CUDA]) as prof:
        for _ in range(20):
            fn()
        torch.cuda.synchronize()
    print(f"\n==== {name}: top CUDA ops (seq={N}) ====")
    print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=5))

## What to paste back

The **cell 6** (correctness) and **cell 8** (benchmark table + `Best vs naive fp16` line) output. That best number is the honest, measured figure for the resume bullet. If it beats naive, we word the bullet to it; if you also want an end-to-end tok/s lift, I'll add a cell that wires this kernel into GPT-2's attention.